In [1]:
import re
import numpy as np
import pandas as pd

# Cargar el dataset preprocesado por tu compañero
df = pd.read_csv('../data/interim/plazavea_preprocesado.csv')

print(f'Dimensiones del dataset cargado: {df.shape}')
df.head(3)

Dimensiones del dataset cargado: (1491, 17)


,id_producto,nombre,marca,categoria,precio_venta,precio_regular,tiene_precio_oh,descuento_pct,url_producto,fecha_extraccion,tiene_descuento,ahorro_soles,es_outlier_precio,macro_grupo,es_marca_propia,marca_agrupada,precio_venta_log
0,galleta-oreo-regular-paquete-6un,Galletas OREO Original Paquete 6un,OREO,Abarrotes,4.7,5.2,False,9.62,https://www.plazavea.com.pe/galleta-oreo-regul...,2026-09-22 20:22:48,True,0.5,False,Envasados,False,OREO,1.740466
1,minis-bombones-de-chocolate-bon-o-bon-multipac...,Minis Bombones de Chocolate BON O BON Multipac...,BON O BON,Abarrotes,26.0,26.0,False,0.00,https://www.plazavea.com.pe/minis-bombones-de-...,2026-09-22 20:22:48,False,0.0,True,Envasados,False,OTRAS,3.295837
2,galleta-chocobum-empaque-33g-paquete-6un,Galleta CHOCOBUM Empaque 33g Paquete 6un,CHOCOBUM,Abarrotes,4.2,5.3,False,20.75,https://www.plazavea.com.pe/galleta-chocobum-e...,2026-09-22 20:22:48,True,1.1,False,Envasados,False,OTRAS,1.648659


In [2]:
def extraer_contenido_y_precio(row):
  nombre = str(row['nombre']).lower()
  precio = row['precio_venta']

  # Patrones comunes en nombres de supermercados
  # 1. Buscar gramos (ej: 500g, 500 g)
  match_g = re.search(r'(\d+[\.,]?\d*)\s*(?:gr|g)\b', nombre)
  # 2. Buscar kilogramos (ej: 1kg, 1.5 kg)
  match_kg = re.search(r'(\d+[\.,]?\d*)\s*(?:kg|kilo)\b', nombre)
  # 3. Buscar mililitros (ej: 500ml, 500 ml)
  match_ml = re.search(r'(\d+[\.,]?\d*)\s*(?:ml|cc)\b', nombre)
  # 4. Buscar litros (ej: 1lt, 1 l, 1.5l)
  match_l = re.search(r'(\d+[\.,]?\d*)\s*(?:lt|l)\b', nombre)
  # 5. Buscar unidades o paquetes (ej: 6un, 6 un, 6pack)
  match_un = re.search(r'(\d+[\.,]?\d*)\s*(?:un|und|pack|pq|paquete)\b', nombre)

  cantidad_base = np.nan
  tipo_unidad = 'desconocido'
  precio_unitario_normalizado = np.nan

  if match_kg:
    val = float(match_kg.group(1).replace(',', '.'))
    cantidad_base = val * 1000  # Convertir a gramos
    tipo_unidad = '100g'
    precio_unitario_normalizado = (precio / cantidad_base) * 100
  elif match_g:
    val = float(match_g.group(1).replace(',', '.'))
    cantidad_base = val
    tipo_unidad = '100g'
    precio_unitario_normalizado = (precio / cantidad_base) * 100
  elif match_l:
    val = float(match_l.group(1).replace(',', '.'))
    cantidad_base = val * 1000  # Convertir a mililitros
    tipo_unidad = '100ml'
    precio_unitario_normalizado = (precio / cantidad_base) * 100
  elif match_ml:
    val = float(match_ml.group(1).replace(',', '.'))
    cantidad_base = val
    tipo_unidad = '100ml'
    precio_unitario_normalizado = (precio / cantidad_base) * 100
  elif match_un:
    val = float(match_un.group(1).replace(',', '.'))
    cantidad_base = val
    tipo_unidad = 'unidad'
    precio_unitario_normalizado = precio / cantidad_base

  return pd.Series([
      cantidad_base,
      tipo_unidad,
      round(precio_unitario_normalizado, 4) if not np.isnan(precio_unitario_normalizado) else np.nan
  ])


# Aplicar la función al DataFrame
df[['cantidad_base', 'tipo_unidad_medida', 'precio_unitario_normalizado']] = df.apply(
    extraer_contenido_y_precio, axis=1
)

# Vista previa de las nuevas variables
df[['nombre', 'precio_venta', 'tipo_unidad_medida', 'precio_unitario_normalizado']].head(10)

,nombre,precio_venta,tipo_unidad_medida,precio_unitario_normalizado
0,Galletas OREO Original Paquete 6un,4.70,unidad,0.7833
1,Minis Bombones de Chocolate BON O BON Multipac...,26.00,desconocido,NaN
2,Galleta CHOCOBUM Empaque 33g Paquete 6un,4.20,100g,12.7273
3,Galletas OREO Original Paquete 12un,8.40,unidad,0.7000
4,Gomas AMBROSOLI Cerebros Bolsa 250g,14.50,100g,5.8000
5,Caramelos ARCOR Sangrientos Mora Bolsa 312g,9.10,100g,2.9167
6,Chupetín de Halloween MISTER POPS Extreme Duo ...,10.80,desconocido,NaN
7,Chupetines MISTER POPS Ciclopops Halloween Bol...,10.80,desconocido,NaN
8,Galletas OREO Sabor Vainilla Paquete 6un,4.79,unidad,0.7983
9,Galletas OREO Sabor a Fresa Paquete 6un,4.79,unidad,0.7983


In [3]:
def clasificar_descuento(pct):
  if pct == 0:
    return 'Sin descuento'
  elif pct < 10:
    return 'Bajo (<10%)'
  elif pct <= 25:
    return 'Moderado (10%-25%)'
  else:
    return 'Alto (>25%)'


df['rango_descuento'] = df['descuento_pct'].apply(clasificar_descuento)

# Ver distribución de los descuentos clasificados
print(df['rango_descuento'].value_counts())

rango_descuento
Sin descuento         773
Moderado (10%-25%)    366
Bajo (<10%)           249
Alto (>25%)           103
Name: count, dtype: int64


In [4]:
# Ejemplo: Promedio de precio por 100g o unidad agrupado por categoría
resumen_unitario = df.groupby(['categoria', 'tipo_unidad_medida'])['precio_unitario_normalizado'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).reset_index()

print(resumen_unitario.head(10))

           categoria tipo_unidad_medida  count      mean   median     min  \
0          Abarrotes               100g     90  5.486993  2.54150  0.2600   
1          Abarrotes              100ml      8  0.753613  0.78885  0.2890   
2          Abarrotes        desconocido      0       NaN      NaN     NaN   
3          Abarrotes             unidad     23  1.026178  0.86670  0.0790   
4            Bebidas               100g      3  7.794867  7.69230  7.6923   
5            Bebidas              100ml    103  1.335373  0.50000  0.0714   
6            Bebidas        desconocido      0       NaN      NaN     NaN   
7            Bebidas             unidad      2  8.375000  8.37500  6.1500   
8  Carnes y Pescados               100g     35  3.378923  1.99000  0.9600   
9  Carnes y Pescados        desconocido      0       NaN      NaN     NaN   

       max  
0  43.5000  
1   1.0556  
2      NaN  
3   3.0000  
4   8.0000  
5  14.0563  
6      NaN  
7  10.6000  
8   9.9000  
9      NaN  


In [5]:
# Guardar en la carpeta oficial de datos procesados del proyecto
ruta_salida = '../data/processed/plazavea_final.csv'
df.to_csv(ruta_salida, index=False)
print(f'Dataset final exportado exitosamente en: {ruta_salida}')

Dataset final exportado exitosamente en: ../data/processed/plazavea_final.csv
